# Chimera v2.4 -- Validation & Ablation Analysis

This notebook runs the full validation and ablation suite for Chimera v2.4,
producing all comparison tables and supplementary figures for the manuscript.

**Sections:**
1. **Baseline Ladder** -- Train or load scVI, Harmony, scANVI baselines and Chimera configs; compute scIB metrics and cross-species label transfer.
2. **Ablation Suite** -- Train all priority ablation configurations (A, D, F, M, J, C, H, B') and compare.
3. **Alpha Analysis** -- Examine alpha distributions, delta-alpha two-sidedness across configs.
4. **Stress Tests** -- Gene dropout and conservation-prior shuffle robustness checks.
5. **scVI Post-hoc Comparison** -- Correlate Chimera alpha with scVI-derived per-gene F-statistics.

**Prerequisites:**
- Notebook `01_data_preparation` has been run (prepared AnnData saved to disk).
- Notebook `02_training` has been run (trained Chimera model checkpoint available).

**Expected runtime:** ~8--12 hours for the full ablation suite on a single A100 GPU.  
Individual sections can be run independently if checkpoints are available.

In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import spearmanr

# Chimera imports
sys.path.insert(0, str(Path(".").resolve().parent))
from chimera._model import ChimeraModel
from chimera._constants import ABLATION_CONFIGS
from scripts.train_baselines import (
    train_baseline_ladder,
    compute_scvi_f_statistics,
    compare_fstats_with_alpha,
)
from scripts.compute_metrics import (
    compute_all_metrics,
    cross_species_label_transfer,
    _compute_scib_metrics,
)
from scripts.run_ablations import run_ablation_suite, run_single_ablation

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="scvi")

# Plotting defaults
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "figure.figsize": (8, 5),
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.1,
})
sns.set_style("whitegrid")

# Paths
DATA_DIR = Path("..") / "data"
RESULTS_DIR = Path("..") / "results" / "validation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_DIR = RESULTS_DIR / "baselines"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
ABLATION_DIR = RESULTS_DIR / "ablations"
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = RESULTS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "camex_pancreas"
ADATA_PATH = DATA_DIR / f"{DATASET_NAME}_prepared.h5ad"

SEED = 42
MAX_EPOCHS = 200

print(f"Data path:    {ADATA_PATH}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Seed:         {SEED}")
print(f"Max epochs:   {MAX_EPOCHS}")

In [ ]:
adata = ad.read_h5ad(ADATA_PATH)
print(f"Loaded AnnData: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Species: {adata.obs['species'].value_counts().to_dict()}")
print(f"Cell types: {adata.obs['cell_type'].nunique()} unique")
print(f"Batches: {adata.obs['batch'].nunique()} unique")
print(f"Gene features shape: {adata.varm['gene_features'].shape}")
print(f"Conservation prior range: [{adata.var['conservation_prior'].min():.3f}, {adata.var['conservation_prior'].max():.3f}]")

---
## Section 1: Baseline Ladder

Train or load baseline methods (scVI, Harmony, scANVI) alongside Chimera ablation
configs D and F.  Compute the full scIB metric suite plus cross-species label
transfer accuracy for all methods.

In [ ]:
# Train baselines (or load from checkpoints if they exist)
baseline_results_path = BASELINE_DIR / "training_results.json"
adata_with_latents_path = BASELINE_DIR / "adata_with_latents.h5ad"

if adata_with_latents_path.exists():
    print("Loading pre-computed baseline results ...")
    adata_eval = ad.read_h5ad(adata_with_latents_path)
    with open(baseline_results_path) as f:
        baseline_results = json.load(f)
    print(f"Loaded: {list(baseline_results.keys())}")
else:
    print("Training baseline ladder from scratch (this may take several hours) ...")
    baseline_results = train_baseline_ladder(
        dataset_path=str(ADATA_PATH),
        output_dir=str(BASELINE_DIR),
        seed=SEED,
        max_epochs=MAX_EPOCHS,
        wandb_project="chimera-validation",
    )
    adata_eval = ad.read_h5ad(adata_with_latents_path)
    print(f"Baselines trained: {list(baseline_results.keys())}")

# List available latent representations
latent_keys = [k for k in adata_eval.obsm.keys() if k.startswith("X_")]
print(f"Available latent keys: {latent_keys}")

In [ ]:
# Compute the full scIB metric suite for every latent representation
metrics_csv_path = RESULTS_DIR / "baseline_comparison.csv"

if metrics_csv_path.exists():
    print("Loading pre-computed metrics ...")
    comparison_table = pd.read_csv(metrics_csv_path)
else:
    print("Computing metrics for all methods ...")
    comparison_table = compute_all_metrics(
        adata_path=str(adata_with_latents_path),
        latent_keys=latent_keys,
        output_path=str(metrics_csv_path),
        batch_key="species",
        label_key="cell_type",
    )

# Clean up method names for display
comparison_table["method"] = comparison_table["method"].str.replace("X_", "")
print(comparison_table.to_string(index=False))

In [ ]:
# Figure 1: Bar chart comparing methods across key metrics
key_metrics = ["batch_ASW", "cell_type_ASW", "NMI", "ARI", "cross_species_transfer_acc"]
available_metrics = [m for m in key_metrics if m in comparison_table.columns]

plot_df = comparison_table.set_index("method")[available_metrics].copy()

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(plot_df.index))
width = 0.15
colors = sns.color_palette("Set2", len(available_metrics))

for i, metric in enumerate(available_metrics):
    offset = (i - len(available_metrics) / 2 + 0.5) * width
    bars = ax.bar(x + offset, plot_df[metric].values, width, label=metric, color=colors[i])
    for bar, val in zip(bars, plot_df[metric].values):
        if np.isfinite(val):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{val:.2f}",
                ha="center", va="bottom", fontsize=6, rotation=45,
            )

ax.set_xticks(x)
ax.set_xticklabels(plot_df.index, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("Baseline Comparison: Key Integration Metrics")
ax.legend(loc="upper left", fontsize=8, ncol=2)
ax.set_ylim(0, 1.15)
sns.despine()
fig.tight_layout()

fig.savefig(FIGURE_DIR / "fig1_baseline_comparison_bar.pdf")
fig.savefig(FIGURE_DIR / "fig1_baseline_comparison_bar.png")
print(f"Saved: {FIGURE_DIR / 'fig1_baseline_comparison_bar.pdf'}")
plt.show()

---
## Section 2: Ablation Suite

Run the priority ablation configurations to quantify the contribution of each
Chimera v2.4 component.  Configs tested:

| Config | Description |
|--------|-------------|
| A      | Full v2.4 (default) |
| D      | No anchor (lambda_anchor=0) |
| F      | No adversarial (adv_weight=0) |
| M      | Per-gene learnable alpha (no GIN) |
| J      | No FQR (lambda_fit=0) |
| C      | No L1 sparsity (lambda_res=0) |
| H      | Higher lambda_aux=0.5 |
| B'     | Alpha frozen at conservation prior |

In [ ]:
# Run ablation suite (or load from checkpoints)
ablation_configs = ["A", "D", "F", "M", "J", "C", "H", "B_prime"]
ablation_summary_path = ABLATION_DIR / "ablation_summary.csv"

if ablation_summary_path.exists():
    print("Loading pre-computed ablation results ...")
    ablation_df = pd.read_csv(ablation_summary_path)
else:
    print(f"Running ablation suite for configs: {ablation_configs}")
    print("This will take several hours ...")
    ablation_df = run_ablation_suite(
        dataset_path=str(ADATA_PATH),
        configs=ablation_configs,
        output_dir=str(ABLATION_DIR),
        max_epochs=MAX_EPOCHS,
        seed=SEED,
        species_key="species",
        batch_key="batch",
    )

print(f"\nAblation results ({len(ablation_df)} configs):")
print(ablation_df.to_string(index=False))

In [ ]:
# Ablation comparison table with config descriptions
config_descriptions = {
    name: cfg.get("note", "") for name, cfg in ABLATION_CONFIGS.items()
}

display_df = ablation_df.copy()
display_df["description"] = display_df["config"].map(config_descriptions)

# Reorder columns for readability
metric_cols = [
    col for col in display_df.columns
    if col not in ["config", "description", "status", "wall_time_s", "error"]
]
display_cols = ["config", "description", "status", "wall_time_s"] + metric_cols
display_df = display_df[[c for c in display_cols if c in display_df.columns]]

# Save the annotated table
display_df.to_csv(RESULTS_DIR / "ablation_comparison_annotated.csv", index=False)
print(display_df.to_string(index=False))

In [ ]:
# Figure 2: Heatmap of ablation configs x metrics
heatmap_metrics = [
    "alpha_mean", "alpha_std", "alpha_frac_below_0.1", "alpha_frac_above_0.9",
    "delta_alpha_frac_positive", "delta_alpha_frac_negative", "delta_alpha_abs_mean",
    "latent_mean_var", "latent_effective_dims",
]
available_heatmap_metrics = [m for m in heatmap_metrics if m in ablation_df.columns]

heatmap_data = ablation_df.set_index("config")[available_heatmap_metrics].astype(float)

# Normalize each column to [0, 1] for color mapping
heatmap_norm = heatmap_data.copy()
for col in heatmap_norm.columns:
    col_min = heatmap_norm[col].min()
    col_max = heatmap_norm[col].max()
    if col_max - col_min > 1e-10:
        heatmap_norm[col] = (heatmap_norm[col] - col_min) / (col_max - col_min)
    else:
        heatmap_norm[col] = 0.5

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    heatmap_norm,
    annot=heatmap_data.round(3),
    fmt="",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Normalized value"},
)
ax.set_title("Ablation Suite: Configs x Metrics")
ax.set_ylabel("Ablation config")
ax.set_xlabel("Metric")
plt.xticks(rotation=35, ha="right")
fig.tight_layout()

fig.savefig(FIGURE_DIR / "fig2_ablation_heatmap.pdf")
fig.savefig(FIGURE_DIR / "fig2_ablation_heatmap.png")
print(f"Saved: {FIGURE_DIR / 'fig2_ablation_heatmap.pdf'}")
plt.show()

---
## Section 3: Alpha Analysis

Compare the learned alpha distributions across ablation configurations to understand
how each component shapes gene-level integration weights.

Key comparison: config **D** (no anchor, bidirectional delta-alpha expected)
vs config **J** (no FQR, one-directional delta-alpha -- FM4 failure mode).

In [ ]:
# Load alpha and delta_alpha arrays for each ablation config
alpha_configs = ["D", "F", "M", "J", "A", "B_prime"]
alpha_dict = {}
delta_alpha_dict = {}
conservation_prior = adata.var["conservation_prior"].values

for cfg_name in alpha_configs:
    alpha_path = ABLATION_DIR / f"config_{cfg_name}" / "alpha.npy"
    delta_path = ABLATION_DIR / f"config_{cfg_name}" / "delta_alpha.npy"
    if alpha_path.exists():
        alpha_dict[cfg_name] = np.load(alpha_path)
        if delta_path.exists():
            delta_alpha_dict[cfg_name] = np.load(delta_path)
        else:
            delta_alpha_dict[cfg_name] = alpha_dict[cfg_name] - conservation_prior
        print(f"Config {cfg_name}: alpha mean={alpha_dict[cfg_name].mean():.4f}, "
              f"std={alpha_dict[cfg_name].std():.4f}, "
              f"delta_alpha frac_pos={(delta_alpha_dict[cfg_name] > 0).mean():.3f}")
    else:
        print(f"Config {cfg_name}: alpha.npy not found, skipping.")

print(f"\nLoaded alpha for {len(alpha_dict)} configs: {list(alpha_dict.keys())}")

In [ ]:
# Alpha distribution comparison across configs
n_configs = len(alpha_dict)
if n_configs > 0:
    fig, axes = plt.subplots(1, n_configs, figsize=(3.5 * n_configs, 4), sharey=True)
    if n_configs == 1:
        axes = [axes]

    for ax, (cfg_name, alpha_vals) in zip(axes, alpha_dict.items()):
        ax.hist(alpha_vals, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
        ax.axvline(alpha_vals.mean(), color="red", linestyle="--", linewidth=1.5,
                   label=f"mean={alpha_vals.mean():.3f}")
        ax.axvline(np.median(alpha_vals), color="orange", linestyle=":", linewidth=1.5,
                   label=f"median={np.median(alpha_vals):.3f}")
        ax.set_xlabel("Alpha")
        ax.set_title(f"Config {cfg_name}")
        ax.legend(fontsize=7)
        ax.set_xlim(0, 1)

    axes[0].set_ylabel("Gene count")
    fig.suptitle("Alpha Distributions Across Ablation Configs", y=1.02, fontsize=13)
    sns.despine()
    fig.tight_layout()

    fig.savefig(FIGURE_DIR / "fig3_alpha_distributions.pdf")
    fig.savefig(FIGURE_DIR / "fig3_alpha_distributions.png")
    print(f"Saved: {FIGURE_DIR / 'fig3_alpha_distributions.pdf'}")
    plt.show()
else:
    print("No alpha arrays available for plotting.")

In [ ]:
# Delta-alpha two-sidedness: Config D (bidirectional) vs Config J (one-directional)
# Config D: no anchor -- GIN corrections should still be bidirectional with FQR active
# Config J: no FQR -- FM4 predicts delta_alpha > 0 is unreachable (only negative corrections)

compare_configs = ["D", "J"]
available_compare = [c for c in compare_configs if c in delta_alpha_dict]

if len(available_compare) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    for ax, cfg_name in zip(axes, available_compare):
        da = delta_alpha_dict[cfg_name]
        frac_pos = (da > 0).mean()
        frac_neg = (da < 0).mean()

        # Color positive and negative separately
        ax.hist(da[da >= 0], bins=40, color="#2196F3", alpha=0.8,
                label=f"Positive ({frac_pos:.1%})", edgecolor="white")
        ax.hist(da[da < 0], bins=40, color="#F44336", alpha=0.8,
                label=f"Negative ({frac_neg:.1%})", edgecolor="white")
        ax.axvline(0, color="black", linewidth=1.2, linestyle="-")
        ax.axvline(da.mean(), color="gold", linewidth=1.5, linestyle="--",
                   label=f"mean={da.mean():.4f}")
        ax.set_xlabel("Delta-alpha (alpha - prior)")
        ax.set_title(f"Config {cfg_name}: {ABLATION_CONFIGS[cfg_name].get('note', '')}")
        ax.legend(fontsize=8)

    axes[0].set_ylabel("Gene count")
    fig.suptitle("Delta-Alpha Two-Sidedness: D (bidirectional) vs J (one-directional, no FQR)",
                 y=1.02, fontsize=12)
    sns.despine()
    fig.tight_layout()

    fig.savefig(FIGURE_DIR / "fig4_delta_alpha_D_vs_J.pdf")
    fig.savefig(FIGURE_DIR / "fig4_delta_alpha_D_vs_J.png")
    print(f"Saved: {FIGURE_DIR / 'fig4_delta_alpha_D_vs_J.pdf'}")
    plt.show()

    # Quantitative summary
    for cfg_name in available_compare:
        da = delta_alpha_dict[cfg_name]
        print(f"\nConfig {cfg_name}:")
        print(f"  frac positive: {(da > 0).mean():.4f}")
        print(f"  frac negative: {(da < 0).mean():.4f}")
        print(f"  mean positive: {da[da > 0].mean():.4f}" if (da > 0).any() else "  mean positive: N/A")
        print(f"  mean negative: {da[da < 0].mean():.4f}" if (da < 0).any() else "  mean negative: N/A")
        print(f"  abs mean:      {np.abs(da).mean():.4f}")
else:
    print(f"Need both configs D and J for comparison; available: {available_compare}")

---
## Section 4: Stress Tests

Two robustness checks to verify Chimera v2.4 gracefully handles data corruption:

1. **Gene dropout test**: Randomly zero out 20% of genes and check that integration
   metrics do not catastrophically degrade.
2. **Conservation prior shuffle**: Randomize the conservation prior so it carries
   no information, and verify that alpha can still learn from data (FQR should
   compensate).

In [ ]:
# Stress Test 1: Gene Dropout
# Randomly zero 20% of genes per cell and retrain, then compare metrics
print("=== Stress Test 1: Gene Dropout (20%) ===")

rng = np.random.RandomState(SEED)
adata_dropout = adata.copy()

# Create a dropout mask: for each cell, zero out 20% of genes
n_cells, n_genes = adata_dropout.X.shape
dropout_frac = 0.20
n_dropout = int(n_genes * dropout_frac)

import scipy.sparse as sp
X = adata_dropout.X
if sp.issparse(X):
    X_dense = np.array(X.todense())
else:
    X_dense = np.array(X, copy=True)

for i in range(n_cells):
    drop_idx = rng.choice(n_genes, n_dropout, replace=False)
    X_dense[i, drop_idx] = 0

adata_dropout.X = sp.csr_matrix(X_dense)
print(f"Applied {dropout_frac:.0%} gene dropout to {n_cells} cells ({n_dropout} genes per cell)")
print(f"Original total counts: {adata.X.sum():.0f}")
print(f"Dropout total counts:  {adata_dropout.X.sum():.0f}")

# Train Chimera config A on dropout data
dropout_dir = RESULTS_DIR / "stress_tests" / "gene_dropout"
dropout_dir.mkdir(parents=True, exist_ok=True)
dropout_model_path = dropout_dir / "config_A"

ChimeraModel.setup_anndata(adata_dropout, species_key="species", batch_key="batch")
model_dropout = ChimeraModel.from_config(adata_dropout, "A")
model_dropout.train(max_epochs=MAX_EPOCHS)

latent_dropout = model_dropout.get_latent_representation()
alpha_dropout = model_dropout.get_alpha()

# Store latent for metric computation
adata_dropout.obsm["X_Chimera_dropout"] = latent_dropout
np.save(dropout_dir / "alpha_dropout.npy", alpha_dropout)

print(f"\nDropout model alpha: mean={alpha_dropout.mean():.4f}, std={alpha_dropout.std():.4f}")
print(f"Dropout model latent dims with var >= 0.1: {(latent_dropout.var(axis=0) >= 0.1).sum()}/{latent_dropout.shape[1]}")

In [ ]:
# Compute metrics for dropout model and compare with clean model
dropout_metrics = _compute_scib_metrics(
    adata_dropout,
    embed_key="X_Chimera_dropout",
    batch_key="species",
    label_key="cell_type",
)
dropout_xfer = cross_species_label_transfer(
    adata_dropout,
    embed_key="X_Chimera_dropout",
    species_key="species",
    label_key="cell_type",
)
dropout_metrics["cross_species_transfer_acc"] = dropout_xfer.get("mean_accuracy", float("nan"))

# Load clean config A metrics from ablation results for comparison
clean_row = ablation_df[ablation_df["config"] == "A"].iloc[0] if "A" in ablation_df["config"].values else None

print("\nGene Dropout Stress Test Results:")
print(f"{'Metric':<30} {'Dropout':>10} {'Note':>20}")
print("-" * 62)
for metric, value in sorted(dropout_metrics.items()):
    print(f"{metric:<30} {value:>10.4f}")

# Save
pd.DataFrame([dropout_metrics]).to_csv(dropout_dir / "dropout_metrics.csv", index=False)
print(f"\nSaved to {dropout_dir / 'dropout_metrics.csv'}")

In [ ]:
# Stress Test 2: Conservation Prior Shuffle
# Randomize the conservation prior so it carries no information.
# Alpha should still learn from data via FQR.
print("=== Stress Test 2: Conservation Prior Shuffle ===")

adata_shuffled = adata.copy()
original_prior = adata_shuffled.var["conservation_prior"].values.copy()
shuffled_prior = rng.permutation(original_prior)
adata_shuffled.var["conservation_prior"] = shuffled_prior

print(f"Original prior: mean={original_prior.mean():.4f}, std={original_prior.std():.4f}")
print(f"Shuffled prior: mean={shuffled_prior.mean():.4f}, std={shuffled_prior.std():.4f}")
print(f"Spearman(original, shuffled) = {spearmanr(original_prior, shuffled_prior).statistic:.4f}")

# Train Chimera config A on shuffled prior
shuffle_dir = RESULTS_DIR / "stress_tests" / "prior_shuffle"
shuffle_dir.mkdir(parents=True, exist_ok=True)

ChimeraModel.setup_anndata(adata_shuffled, species_key="species", batch_key="batch")
model_shuffled = ChimeraModel.from_config(adata_shuffled, "A")
model_shuffled.train(max_epochs=MAX_EPOCHS)

alpha_shuffled = model_shuffled.get_alpha()

# Compare learned alpha from shuffled prior vs original alpha from config A
alpha_clean_path = ABLATION_DIR / "config_A" / "alpha.npy"
if alpha_clean_path.exists():
    alpha_clean = np.load(alpha_clean_path)
    rho_clean_shuffled, p_clean_shuffled = spearmanr(alpha_clean, alpha_shuffled)
    print(f"\nSpearman(alpha_clean, alpha_shuffled) = {rho_clean_shuffled:.4f} (p={p_clean_shuffled:.2e})")
    print("Interpretation: if FQR works, alpha_shuffled should still correlate with")
    print("alpha_clean because the data signal overcomes the uninformative prior.")
else:
    alpha_clean = None
    print("Clean config A alpha not found; skipping direct comparison.")

np.save(shuffle_dir / "alpha_shuffled.npy", alpha_shuffled)
print(f"\nShuffled model alpha: mean={alpha_shuffled.mean():.4f}, std={alpha_shuffled.std():.4f}")

# Save summary
summary = {
    "alpha_shuffled_mean": float(alpha_shuffled.mean()),
    "alpha_shuffled_std": float(alpha_shuffled.std()),
    "spearman_clean_vs_shuffled": float(rho_clean_shuffled) if alpha_clean is not None else None,
    "spearman_pvalue": float(p_clean_shuffled) if alpha_clean is not None else None,
}
with open(shuffle_dir / "prior_shuffle_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved to {shuffle_dir / 'prior_shuffle_summary.json'}")

---
## Section 5: scVI Post-hoc Comparison

Compare Chimera's learned alpha with scVI-derived per-gene F-statistics.
High F-statistic means high between-species variance (divergent gene);
low alpha means the model treats the gene as divergent.  

**Expected:** negative Spearman correlation between F-stat and alpha,
confirming that Chimera's learned gene weights agree with an independent
measure of cross-species divergence.

In [ ]:
# Load scVI model for F-statistic computation
import scvi as scvi_pkg

scvi_model_dir = BASELINE_DIR / "scvi_model"

if scvi_model_dir.exists():
    adata_fstat = adata.copy()
    scvi_pkg.model.SCVI.setup_anndata(adata_fstat, batch_key="species")
    scvi_model = scvi_pkg.model.SCVI.load(str(scvi_model_dir), adata=adata_fstat)
    print("Loaded scVI model from checkpoint.")

    # Compute per-gene F-statistics
    f_stats = compute_scvi_f_statistics(adata_fstat, scvi_model)
    print(f"F-statistics: shape={f_stats.shape}, median={np.median(f_stats):.2f}, "
          f"max={f_stats.max():.2f}")
else:
    print(f"scVI model not found at {scvi_model_dir}. Training from scratch ...")
    adata_fstat = adata.copy()
    scvi_pkg.model.SCVI.setup_anndata(adata_fstat, batch_key="species")
    scvi_model = scvi_pkg.model.SCVI(adata_fstat, n_latent=10)
    scvi_model.train(max_epochs=MAX_EPOCHS, early_stopping=True)
    scvi_model.save(str(scvi_model_dir), overwrite=True)
    f_stats = compute_scvi_f_statistics(adata_fstat, scvi_model)
    print(f"Trained scVI and computed F-statistics: shape={f_stats.shape}")

In [ ]:
# Compare Chimera alpha (config A) with scVI F-statistics
alpha_a_path = ABLATION_DIR / "config_A" / "alpha.npy"

if alpha_a_path.exists():
    alpha_a = np.load(alpha_a_path)
else:
    # Fallback: use alpha from any available Chimera model
    available_alpha_configs = list(alpha_dict.keys())
    fallback_cfg = available_alpha_configs[0] if available_alpha_configs else None
    if fallback_cfg:
        alpha_a = alpha_dict[fallback_cfg]
        print(f"Config A alpha not found; using config {fallback_cfg} as fallback.")
    else:
        raise FileNotFoundError("No alpha arrays available for comparison.")

comparison = compare_fstats_with_alpha(f_stats, alpha_a)
print(f"\nSpearman correlation: rho = {comparison['spearman_r']:.4f}")
print(f"P-value:             p = {comparison['p_value']:.2e}")
print(f"N genes used:        n = {comparison['n_genes']}")

In [ ]:
# Figure 5: Scatter plot -- Chimera alpha vs scVI F-statistic
mask = np.isfinite(f_stats) & np.isfinite(alpha_a) & (f_stats > 0)
f_valid = f_stats[mask]
alpha_valid = alpha_a[mask]

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(
    np.log10(f_valid + 1),
    alpha_valid,
    s=4,
    alpha=0.3,
    c="steelblue",
    edgecolors="none",
    rasterized=True,
)

# Trend line (LOWESS)
from scipy.ndimage import uniform_filter1d
sort_idx = np.argsort(np.log10(f_valid + 1))
x_sorted = np.log10(f_valid + 1)[sort_idx]
y_sorted = alpha_valid[sort_idx]
window = max(len(y_sorted) // 50, 10)
y_smooth = uniform_filter1d(y_sorted.astype(float), size=window)
ax.plot(x_sorted, y_smooth, color="red", linewidth=2, label="Smoothed trend")

# Annotation
rho = comparison["spearman_r"]
pval = comparison["p_value"]
ax.text(
    0.02, 0.02,
    f"Spearman rho = {rho:.3f}\np = {pval:.1e}\nn = {comparison['n_genes']} genes",
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="bottom",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
)

ax.set_xlabel("log10(scVI F-statistic + 1)")
ax.set_ylabel("Chimera alpha")
ax.set_title("Chimera Alpha vs scVI Per-Gene F-Statistic")
ax.legend(loc="upper right")
ax.set_ylim(-0.02, 1.02)
sns.despine()
fig.tight_layout()

fig.savefig(FIGURE_DIR / "fig5_alpha_vs_fstat.pdf")
fig.savefig(FIGURE_DIR / "fig5_alpha_vs_fstat.png")
print(f"Saved: {FIGURE_DIR / 'fig5_alpha_vs_fstat.pdf'}")
plt.show()

---
## Save All Results

In [ ]:
# Consolidate and save all validation outputs
print("=" * 60)
print("Validation & Ablation Analysis -- Summary")
print("=" * 60)

# Save consolidated comparison table
comparison_table.to_csv(RESULTS_DIR / "baseline_comparison.csv", index=False)
print(f"  Baseline comparison: {RESULTS_DIR / 'baseline_comparison.csv'}")

# Save ablation table
ablation_df.to_csv(RESULTS_DIR / "ablation_summary.csv", index=False)
print(f"  Ablation summary:    {RESULTS_DIR / 'ablation_summary.csv'}")

# Save F-stat comparison
fstat_results = {
    "spearman_r": comparison["spearman_r"],
    "p_value": comparison["p_value"],
    "n_genes": comparison["n_genes"],
}
with open(RESULTS_DIR / "fstat_alpha_comparison.json", "w") as f:
    json.dump(fstat_results, f, indent=2)
print(f"  F-stat comparison:   {RESULTS_DIR / 'fstat_alpha_comparison.json'}")

# List all saved figures
figures = sorted(FIGURE_DIR.glob("*.pdf"))
print(f"\nFigures saved ({len(figures)}):")
for fig_path in figures:
    print(f"  {fig_path.name}")

# List all saved tables
tables = sorted(RESULTS_DIR.glob("*.csv"))
print(f"\nTables saved ({len(tables)}):")
for tbl_path in tables:
    print(f"  {tbl_path.name}")

print("\nValidation notebook complete.")